In [3]:
import pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import NearestNeighbors

train = pd.read_csv("fill/train.csv")
test  = pd.read_csv("fill/test.csv")
y = train["stress_score"].values
feat_cols = [c for c in train.columns if c not in ["stress_score"]]

# 1) 완전 동일행 교집합(정확 일치)
key_train = train[feat_cols].astype(str).agg("|".join, axis=1)
key_test  = test[feat_cols].astype(str).agg("|".join, axis=1)
overlap = set(key_train).intersection(set(key_test))
print("정확히 같은 행 갯수:", len(overlap))

# 2) 소수점 반올림으로 근접-중복 잡기 (수치형만)
num_cols = train[feat_cols].select_dtypes(include=[np.number]).columns
cat_cols = [c for c in feat_cols if c not in num_cols]
for r in [0,1,2]:
    tt = test.copy()
    tr = train.copy()
    if len(num_cols):
        tr[num_cols] = tr[num_cols].round(r)
        tt[num_cols] = tt[num_cols].round(r)
    ktr = tr[feat_cols].astype(str).agg("|".join, axis=1)
    kte = tt[feat_cols].astype(str).agg("|".join, axis=1)
    print(f"라운드 {r}자리 교집합:", len(set(ktr).intersection(set(kte))))

# 3) 근접 이웃 거리 확인(표준화+원핫 후 유클리드)
ct = ColumnTransformer([
    ("num", StandardScaler(), list(num_cols)),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
], remainder="drop")
Xtr = ct.fit_transform(train[feat_cols])
Xte = ct.transform(test[feat_cols])

nn = NearestNeighbors(n_neighbors=1, metric="euclidean").fit(Xtr)
dist, idx = nn.kneighbors(Xte, n_neighbors=1)
print("테스트→트레인 최근접 거리 통계:", np.min(dist), np.median(dist), np.quantile(dist, [0.9,0.99]))


정확히 같은 행 갯수: 10
라운드 0자리 교집합: 64
라운드 1자리 교집합: 11
라운드 2자리 교집합: 10
테스트→트레인 최근접 거리 통계: 0.0 1.6334426590100284 [3.10845731 3.87399769]
